# RASA — Mission Case Study
Menjalankan ranking dan basket optimization langsung dari artifact, tanpa server atau training ulang.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from rasa.data.schema import load_processed_dataset
from rasa.missions.validation import Mission, Preferences, validate_mission
from rasa.optimization.basket import optimize_basket
from rasa.services.catalog import CatalogService
from rasa.services.model_registry import ModelRegistry

In [ ]:
bundle = load_processed_dataset()
catalog = CatalogService().load_from_bundle(bundle).catalog
registry = ModelRegistry().load()
assert registry.ready, registry.error
mission = Mission(
    goal='Menyiapkan perlengkapan kuliah harian yang minimalis',
    required_categories=['bag', 'stationery', 'drinkware'],
    optional_categories=['organizer'], max_budget=250_000, max_merchants=2,
    preferences=Preferences(style=['minimalist'], colors=['hitam', 'abu'], minimum_rating=4.2),
)
validation = validate_mission(mission, catalog, bundle.categories)
validation.to_dict()

In [ ]:
ranked = registry.ranker.rank(catalog, validation.normalized, strategy='hybrid_mission', top_k=10)
basket = optimize_basket(ranked.scored, validation.normalized, catalog, ranked.diagnostics.get('candidate_generation', {}))
basket.to_dict()

In [ ]:
assert basket.total_price <= mission.max_budget
assert basket.merchant_count <= mission.max_merchants
assert basket.required_category_coverage == 1.0
[(item.product_name, item.category_name, item.price, item.merchant_name) for item in basket.items]